# SVM hyperparameter search with Optuna

In [ ]:
!pip install -q optuna 2>/dev/null

In [ ]:
import os, time, warnings, joblib
import numpy as np
import pandas as pd
import optuna
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.INFO)
print('Imports OK')

In [ ]:
COMPETITION_DATA_ROOT = '/kaggle/input/competitions/birdclef-2026'
FEATURE_DATASET_PATH  = '/kaggle/input/datasets/shuoyiyang/birdclef-2026-features-v2-r3-t3'
OUTPUT_DIR            = '/kaggle/working'

INPUT_FEATURE_PATH = f'{FEATURE_DATASET_PATH}/features_v2_rating2_top3.parquet'
INPUT_LABEL_PATH   = f'{FEATURE_DATASET_PATH}/labels_v2_rating2_top3.parquet'

HPO_TIMEOUT_S       = 8 * 3600   
HPO_TRAIN_FRAC      = 0.50       
HPO_MIN_POS_TRAIN   = 10
HPO_MIN_POS_VAL     = 3
RANDOM_SEED         = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
class SafeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        try:
            if len(self.classes_) < 2:
                raise ValueError('only one class')
            self.model_ = clone(self.base_estimator)
            self.model_.fit(X, y)
        except Exception:
            self.model_ = DummyClassifier(strategy='constant', constant=y[0])
            self.model_.fit(X, y)
        return self
    def predict_proba(self, X):
        return self.model_.predict_proba(X)

In [ ]:
# Load taxonomy + features 
taxonomy_df = pd.read_csv(f'{COMPETITION_DATA_ROOT}/taxonomy.csv')
ALL_SPECIES = taxonomy_df['primary_label'].unique().tolist()

X_df      = pd.read_parquet(INPUT_FEATURE_PATH)
labels_df = pd.read_parquet(INPUT_LABEL_PATH)

mlb = MultiLabelBinarizer(classes=ALL_SPECIES)
y_matrix_full = mlb.fit_transform(labels_df['all_labels'])

counts        = y_matrix_full.sum(axis=0)
keep_indices  = np.where(counts > 0)[0]
ACTIVE_SPECIES = [ALL_SPECIES[i] for i in keep_indices]
y_matrix      = y_matrix_full[:, keep_indices]

X_matrix = X_df.drop(columns=['filename']).values
groups   = X_df['filename'].values

print(f'Samples : {X_matrix.shape[0]:,}')
print(f'Features: {X_matrix.shape[1]}')
print(f'Species : {len(ACTIVE_SPECIES)} active / {len(ALL_SPECIES)} total')

In [ ]:
# Scale, then make a single group-aware train/val split for HPO
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_matrix)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(X_scaled, y_matrix, groups))

rng                = np.random.default_rng(RANDOM_SEED)
hpo_train_subset   = rng.choice(train_idx,
                                size=int(len(train_idx) * HPO_TRAIN_FRAC),
                                replace=False)

X_hpo_train = X_scaled[hpo_train_subset]
y_hpo_train = y_matrix[hpo_train_subset]
X_hpo_val   = X_scaled[val_idx]
y_hpo_val   = y_matrix[val_idx]

species_to_score = [
    j for j in range(y_matrix.shape[1])
    if y_hpo_train[:, j].sum() >= HPO_MIN_POS_TRAIN
    and y_hpo_val[:, j].sum()   >= HPO_MIN_POS_VAL
]

print(f'HPO train subset : {X_hpo_train.shape[0]:,} samples ({HPO_TRAIN_FRAC*100:.0f}% of train)')
print(f'HPO val          : {X_hpo_val.shape[0]:,} samples')
print(f'Species scored   : {len(species_to_score)}')

In [ ]:
# Optuna objective 
def objective(trial):
    C            = trial.suggest_float('C', 1e-3, 1e2, log=True)
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])

    aucs = []
    for j in species_to_score:
        try:
            clf = LinearSVC(
                C=C, class_weight=class_weight, dual=False,
                max_iter=2000, tol=1e-3, random_state=RANDOM_SEED,
            )
            clf.fit(X_hpo_train, y_hpo_train[:, j])
            scores = clf.decision_function(X_hpo_val)
            aucs.append(roc_auc_score(y_hpo_val[:, j], scores))
        except Exception:
            continue
    return float(np.mean(aucs)) if aucs else 0.0

In [ ]:
# Run the study 
STUDY_DB_PATH = os.path.join(OUTPUT_DIR, 'optuna_study_linear.db')
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    study_name='svm_birdclef_linear',
    storage=f'sqlite:///{STUDY_DB_PATH}',
    load_if_exists=True,
)

def progress_cb(study, trial):
    print(f'Trial {trial.number:3d}  value={trial.value:.4f}  '
          f'best={study.best_value:.4f}  params={trial.params}')

t0 = time.time()
study.optimize(objective, timeout=HPO_TIMEOUT_S, callbacks=[progress_cb])
elapsed = time.time() - t0

print('\n' + '=' * 60)
print(f'HPO finished in {elapsed/60:.1f} min  ({len(study.trials)} trials)')
print(f'Best value : {study.best_value:.4f}')
print(f'Best params: {study.best_params}')
print('=' * 60)

In [ ]:
# Build the best estimator 
best = study.best_params

base = LinearSVC(
    C=best['C'], class_weight=best['class_weight'],
    dual=False, max_iter=5000, tol=1e-3,
    random_state=RANDOM_SEED,
)
calibrated = CalibratedClassifierCV(base, cv=5, method='sigmoid')

print('Best estimator built.')
print(f'  C: {best["C"]:.4g}, class_weight: {best["class_weight"]}')

In [ ]:
#Train final model on FULL data 
print(f'Final fit on {X_scaled.shape[0]:,} samples × {X_scaled.shape[1]} features')
print(f'                  × {y_matrix.shape[1]} species')

t0 = time.time()
final_model = OneVsRestClassifier(SafeClassifier(calibrated), n_jobs=-1)
final_model.fit(X_scaled, y_matrix)
print(f'\nFinal training done in {(time.time() - t0)/60:.1f} min')

In [ ]:
model_path   = os.path.join(OUTPUT_DIR, 'svm_model_v2_linear.pkl')
scaler_path  = os.path.join(OUTPUT_DIR, 'svm_scaler_v2_linear.pkl')
species_path = os.path.join(OUTPUT_DIR, 'svm_active_species_v2_linear.pkl')
params_path  = os.path.join(OUTPUT_DIR, 'svm_best_params_v2_linear.pkl')

joblib.dump(final_model,    model_path)
joblib.dump(scaler,         scaler_path)
joblib.dump(ACTIVE_SPECIES, species_path)
joblib.dump(best,           params_path)

for p in (model_path, scaler_path, species_path, params_path):
    print(f'  {p}  ({os.path.getsize(p) / (1024*1024):.1f} MB)')

print(f'\nOptuna study DB also saved at: {STUDY_DB_PATH}')